# R-tag pipeline ΓÇö single run walkthrough

End-to-end **KB331** sample: filter GDS ΓåÆ per-resonator RTEG layouts with MBE/MTE routing.

**Docs:** [README](../README.md) ┬╖ [CLAUDE](../CLAUDE.md) ┬╖ run top-to-bottom from `python_code/`.

| Step | What it does | Main call |
|------|----------------|-----------|
| 1 | Validate inputs | layermap + file check |
| 2 | Find resonators / vias | `identify` |
| 3 | Center resonator in GSG PPD | `prep_resonator_ppd` |
| 4 | Place in die frame + attach preserved interconnect | `prep_rteg_in_frame`, `attach_preserved_filter_interconnect_all` |
| 5.1ΓÇô5.4 | MTE signal: collect ΓåÆ classify ΓåÆ extend ΓåÆ pad stretch | see ┬º5 cells |
| 6.1ΓÇô6.3 | MBE signal + ground filler | see ┬º6 cells |
| export | Complete routed GDS | `export_full_rteg_gds` → `draft_output/route_outputs/` |

**KB331 split (8 resonators):** `center_pad` MTE route ΓåÆ indices **1, 3, 4, 6** ┬╖ `collar_extend` MBE signal ΓåÆ **0, 2, 5, 7**.


In [ ]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ROUTE_OUTPUTS = DRAFT / "route_outputs"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

## Define inputs

Set paths to the filter GDS, die frame, GSG probe template, and Skyworks layermap. All paths are under `input_files/`. The next code cell assigns `FILTER`, `FRAME`, `PPD`, `LAYERMAP`.


In [ ]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process inputs

**~30 s read**

Confirms all four input files exist and prints a short inventory (size + role). Also reads the frame template bbox so you know the probe window size before placement.

**Output:** sanity-check table ΓÇö aborts if anything is missing.


In [ ]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS ΓÇö resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port ΓÇö GSG pad reference",
    "Layer table": "Skyworks name ΓåÆ GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}├ù{fy1 - fy0:.1f} ┬╡m"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


## 2. Selection

**~30 s read**

Prepare for resonator extraction: load the layermap, inspect GDS hierarchy, then identify which instances in the filter become R-tags.

| Sub-step | Module | Purpose |
|----------|--------|---------|
| 2.1 | `layermap.py` | Name Γåö GDS layer/datatype |
| 2.2 | `inspect_refs.py` | Hierarchy / reference listing |
| 2.3 | `separate.py` | Resonator + via identification |

**Output:** `layermap`, `res_df`, `res_list`, `via_df`, `identification`.


### 2.1 ΓÇö `layermap.py`

**~30 s read**

Loads `input_files/layermap` so later steps refer to layers by Skyworks names (`BAW_MBE`, `BAW_MTE`, ΓÇª) instead of raw GDS numbers.

**Call:** `load_layermap(LAYERMAP)` ΓåÆ `layermap` object with `.pair(name)` lookups.


In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

### 2.2 ΓÇö `inspect_refs.py`

**~30 s read**

Walks the filter GDS hierarchy: cell references, labels, and bounding boxes. Quick sanity check that resonator masters and parent cells look like the expected KB331 structure before automated identification.

**Call:** `inspect_gds(FILTER)` (optional for frame/PPD too).


In [ ]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")

### 2.3 ΓÇö `separate.py`

**~30 s read**

Finds placed resonators under the filter parent (`series*`, `shunt*`, `rcap*`, `mimcap*` masters) and `vtb` vias. Returns dataframe rows plus live `Resonator` objects with placement transform preserved.

**Call:** `identify(FILTER)` ΓåÆ `identification` with `.resonator_rows()`, `.resonators`, `.via_rows()`.

**Output:** `res_df`, `res_list` ΓÇö one row/object per R-tag candidate (KB331: 8).


In [ ]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

In [ ]:
via_df

## 3. Separation

**~30 s read**

For each identified resonator, build an in-memory **PPD + resonator** assembly: GSG probe frame from `GSG_frame.gds` with the resonator centered and clearance-adjusted inside it. No die frame yet ΓÇö that is step 4.

**Output:** `ppd_assemblies` ΓÇö one per resonator, ready for frame placement.


### 3 ΓÇö PPD + orientation placement

**~30 s read**

**Calls:** `prep_resonator_ppd` (with `identification` + `layermap`) ΓåÆ optional orientation analysis.

1. **Center** resonator bbox on the PPD template.
2. **Clearance nudge** ΓÇö ΓëÑ10 ┬╡m to GSG pad metal, ΓëÑ6 ┬╡m to release holes.
3. **Orientation shift** ΓÇö small search to maximize pad clearance while keeping DRC-friendly placement.

**Output:** `ppd_assemblies[index].top_cell` ΓÇö PPD ref + resonator ref in a scratch cell.


In [ ]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


### 3 ΓÇö Preview (optional)

**~30 s read**

Optional SVG grid of each PPD+resonator assembly. Use to spot bad centering or pad collisions before die-frame placement. Code is commented out by default.


In [ ]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting up

**~30 s read**

Place each PPD assembly into the **die frame template** (`KB331_N_Frame.gds`) and add the right-side MBE width filler. Margins are measured from the inner MBE ring cavity (not the outer 460├ù580 ┬╡m bbox).

- **X:** PPD/GSG frame left-aligned at 4% inner margin; wide resonators may get a **resonator-only** left shift (5 ┬╡m clearance to filler right edge).
- **Y:** assembly centered in 7% vertical margin band.
- **Filler:** MBE rectangle from inner-frame center line ΓåÆ right margin, full assembly height.

**Output:** `rteg_assemblies` ΓÇö frame + placed PPD/resonator + filler per index.


### 4 Die frame placement

**~30 s read**

**Call:** `prep_rteg_in_frame(ppd_assemblies, FRAME)` ΓåÆ `rteg_assemblies`

PPD and resonator are placed as **separate references** so only the resonator moves when enforcing filler clearance (`resonator_frame_shift` on the assembly object).

**4.1** copies filter `connectMTE` / `connectMBE` metal that touches each resonator into the frame cell (`attach_preserved_filter_interconnect_all`). GDS export is deferred to the final cell at the end of the notebook.


In [ ]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

### 4.1 — Preserved filter interconnect

Copy MTE/MBE polygons from the parent filter `connectMTE` / `connectMBE` cells that overlap each resonator (same rules as step 5.1), shifted into the RTEG frame coordinates, and written onto `assembly.top_cell` before export.

In [ ]:
from src.rteg_collect import (
    attach_preserved_filter_interconnect_all,
    preserved_interconnect_attach_rows,
)

preserved_by_index = attach_preserved_filter_interconnect_all(
    rteg_assemblies,
    res_list,
    identification,
    layermap,
)
inst_by_index = {i: r.inst_name for i, r in enumerate(res_list)}
preserved_attach_df = pd.DataFrame(
    preserved_interconnect_attach_rows(
        preserved_by_index,
        inst_names=inst_by_index,
    )
)
print(f"Attached preserved interconnect for {len(preserved_by_index)} assemblies")
display(preserved_attach_df)

In [ ]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [ ]:
# Intermediate GDS export removed — see Final export (route_outputs/).


---

## 5. Routing (MTE signal)

**~30 s read**

Build per-resonator **MTE (layer 5/0)** signal paths: collect layout roles, decide routing strategy, draw collar lip extensions, then stretch to the center pad when applicable. GDS is written once at the end (`route_outputs/`).


## 5. Routing ΓÇö overview

| Step | Module | What you get |
|------|--------|--------------|
| 5.1 | `rteg_collect` | Ground plates, preserved filter metal, release holes, body MTE |
| 5.2 | `rteg_classify` | Signal vs ground nodes; `mte_route_target` |
| 5.3 | `rteg_mte_extensions` | Preserved filter MTE collar intercepts (no lip draw) |
| 5.4 | `rteg_mte_route` | Stretch preserved MTE to center pad TR/BR corners |
| export | `export_full_rteg_gds` | One complete GDS per index (`route_outputs/`) |

**Routing split:** indices **1, 3, 4, 6** ΓåÆ MTE stretch to center pad ┬╖ **0, 2, 5, 7** ΓåÆ preserved filter MTE only (MBE signal in step 6).


### 5.1 ΓÇö Collect geometry roles

**~30 s read**

**Call:** `collect_geometry_roles(assembly, res, identification, layermap)` ΓåÆ `all_roles[index]`

Snapshots everything routing needs in **RTEG world coordinates**: GSG pad MBE, step-4 filler plate, preserved filter interconnect (MBE/MTE), resonator body metal, release holes, inner frame boundary.

Preserved MTE includes `connectMTE` tabs; series parts may also retain the stadium-adjacent collar (e.g. index 6 ΓåÆ areas 911 + 5191 ┬╡m┬▓).


In [ ]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

### 5.2 ΓÇö Classify GSG nodes

**~30 s read**

**Calls:** `collect_orientation_inputs` ΓåÆ `classify_nodes` ΓåÆ `all_classify[index]`

Labels top/center/bottom GSG nodes as signal or ground and sets **`mte_route_target`**:

- **`center_pad`** ΓÇö mouth tab is closer to the center signal pad than the body center is (route MTE to pad in 5.4).
- **`collar_extend`** ΓÇö otherwise (MBE signal route in 6.1).

KB331: indices 1, 3, 4, 6 = `center_pad`; 0, 2, 5, 7 = `collar_extend`.


In [ ]:
from src.rteg_classify import classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")


### 5.3 ΓÇö Preserved MTE routing setup

**~30 s read**

Filter MTE from step **4.1** is already on the frame. This step selects the mouth collar and records SKILL slope intercepts ΓÇö **no new 14 ┬╡m lip** is drawn.

**Call:** `build_mte_extensions(all_roles, layermap, mte_cfg)` ΓåÆ `all_mte[index]` with preserved collar polygon + intercept metadata for 5.4.

**Output:** `extension_draw.collar_intercept_a/b`; `n_extensions = 0` until 5.4 routes to pad.


In [ ]:
from src.rteg_mte_extensions import (
    MteBuildConfig,
    build_mte_extensions,
    mte_extensions_overview_rows,
    mte_intercept_breakdown_rows,
)

mte_cfg = MteBuildConfig(
    mte_layer="BAW_MTE",
    collar_extension_um=14.0,
    collar_merge_inset_um=4.0,
    skill_pad_expand_um=5.0,
    boolean_precision=1e-3,
)

all_mte = build_mte_extensions(all_roles, layermap, mte_cfg)

inst_names = {idx: roles.inst_name for idx, roles in all_roles.items()}

mte_overview_df = pd.DataFrame(
    mte_extensions_overview_rows(all_mte, inst_names=inst_names)
).sort_values("index")

display(mte_overview_df)
print(f"Prepared preserved MTE routing metadata for {len(all_mte)} resonators")

mte_intercept_df = pd.DataFrame(
    mte_intercept_breakdown_rows(all_mte, inst_names=inst_names)
).sort_values("index")

print("Collar intercepts (SKILL slope A/B on preserved filter MTE):")
display(mte_intercept_df)


### 5.4 ΓÇö Stretch preserved MTE to center signal pad

**~30 s read**

**Call:** `build_mte_pad_routes(all_roles, all_classify, all_mte, layermap, mte_route_cfg)`

**Gate:** `mte_route_target == "center_pad"` only (KB331 indices **1, 3, 4, 6**).

Builds ``mteConn`` from the signal pad **top-right / bottom-right** corners to the **collar–extension junction**, then boolean-merges route + preserved filter MTE.

**Config:** `MteRouteConfig` ΓÇö `pad_touch_overlap_um`, `min_pad_overlap_um2`.


In [ ]:
from src.rteg_mte_route import (
    MteRouteConfig,
    build_mte_pad_routes,
    mte_route_overview_rows,
)

# --- 5.4 tunables (edit here) ---
mte_route_cfg = MteRouteConfig(
    mte_layer="BAW_MTE",
    pad_touch_overlap_um=0.5,
    min_pad_overlap_um2=0.01,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
)

all_mte = build_mte_pad_routes(
    all_roles, all_classify, all_mte, layermap, mte_route_cfg
)

mte_route_df = pd.DataFrame(
    mte_route_overview_rows(all_mte, all_classify, inst_names=inst_names)
).sort_values("index")

display(mte_route_df)
routed_count = int(mte_route_df["routed_to_pad"].sum())
print(f"Routed {routed_count} resonator(s) to center signal pad")


## Step 6 ΓÇö MBE routing

**~30 s read**

When MTE does **not** face the center pad (`collar_extend`), **MBE (layer 2/0)** carries signal (6.1) and ground filler (6.2). When MTE routes to the pad (`center_pad`), step **6.3** routes the step-4 ground filler around MTE keepouts instead.

| Step | Applies to | Purpose |
|------|------------|---------|
| 6.1 | 0, 2, 5, 7 | MBE curve from signal pad ΓåÆ preserved collar |
| 6.2 | 0, 2, 5, 7 | MBE cap + carved ground filler bridge |
| 6.3 | 1, 3, 4, 6 | Carve/route ground filler with MTE clearance |

### 6.1 ΓÇö MBE signal route (pad ΓåÆ collar)

**Gate:** `mbe_extension_applies` → `build_mbe_extensions`

- **`collar_extend` (0, 2, 5, 7):** mouth trace from pad corners to preserved MBE collar.

**Skipped:** indices **1, 3, 4, 6** (`center_pad` / `mte_faces_center`) — step 5.4 already routes MTE to the signal pad.

Output: `all_mbe[index].routed_net`. Run the **6.1 export** cell next to write GDS.


In [ ]:
# MBE pad-to-collar mouth trace — collar_extend only (0, 2, 5, 7); center_pad skipped (5.4 MTE route).

from src.rteg_mbe_extensions import (
    MbeConnectionConfig,
    build_mbe_extensions,
    mbe_extensions_overview_rows,
)

mbe_cfg = MbeConnectionConfig()

all_mbe = build_mbe_extensions(all_roles, all_classify, layermap, mbe_cfg)

mbe_overview_df = pd.DataFrame(
    mbe_extensions_overview_rows(all_mbe, inst_names=inst_names)
).sort_values("index")

display(mbe_overview_df)
mbe_drawn = int(mbe_overview_df["n_extensions"].sum())
print(f"Drew MBE connections for {mbe_drawn} resonators (collar_extend only)")


## Step 6.2 ΓÇö MBE ground body (`collar_extend` only)

**~30 s read**

**Gate:** `mbe_body_collar_extend_applies` ΓåÆ `build_mbe_body_collar_extends` ΓÇö KB331 indices **0, 2, 5, 7**.

1. **MTE extension simplify** ΓÇö reshape only the wild preserved filter `connectMTE` stub: keep the inner boundary chain between two side edges, then close the outer boundary with an axis-aligned right-angle corner outside resonator body MTE.
2. **MBE cap** on outer half of the retraced MTE extension (shifted 3.5 ┬╡m outward onto filler).
3. **Keepouts** ΓÇö body MTE stadium, release holes, retraced MTE extension, 6.1 MBE signal; boolean-carve step-4 filler.
4. **Bridge** ΓÇö reconnect carved filler to cap across stadium gap.
4. **Export** ΓÇö cap + filler pieces as separate `BAW_MBE` polygons; raw step-4 rectangle stripped.

Run the **6.2 export** cell next, then step 6.3 and final export for complete GDS files.


In [ ]:
# MBE ground body for resonators where mte_route_target = collar_extend.

from src.rteg_mbe_body import (
    MbeBodyConfig,
    build_mbe_body_collar_extends,
    mbe_body_overview_rows,
)

COLLAR_EXTEND_INDICES = (0, 2, 5, 7)

mbe_body_cfg = MbeBodyConfig()

all_mbe_body = build_mbe_body_collar_extends(
    all_roles,
    all_classify,
    all_mte,
    all_mbe,
    layermap,
    mbe_body_cfg,
)

mbe_body_overview_df = pd.DataFrame(
    mbe_body_overview_rows(all_mbe_body, inst_names=inst_names)
).sort_values("index")

collar_extend_df = mbe_body_overview_df[
    mbe_body_overview_df["index"].isin(COLLAR_EXTEND_INDICES)
]
display(collar_extend_df)
body_drawn = int(collar_extend_df["n_pieces"].sum())
print(f"Drew MBE ground body for {body_drawn} polygon piece(s) (collar_extend)")


## Step 6.3 ΓÇö MBE ground filler (`center_pad` only)

**~30 s read**

**Gate:** `mbe_body_center_pad_applies` ΓåÆ `build_mbe_body_center_pads` ΓÇö KB331 indices **1, 3, 4, 6**.

Carves the step-4 **MBE width filler** with clearance to MTE body, 5.3 extension, and pad route (layer 5/0). Left edge follows the preserved MBE collar; top/right/bottom are trimmed back from MTE routes (~2 ┬╡m + 1 ┬╡m trim margin) so no pinch corners sit on the extension-to-pad junction.

**Output:** one connected `BAW_MBE` filler polygon per index (collar metal absorbed into filler export).

Results merge into `all_mbe_body` via `merge_mbe_bodies` so empty center_pad placeholders do not overwrite step 6.2 work on indices 0, 2, 5, 7.


In [ ]:
# Step 6.3 ΓÇö center_pad resonators: route the step-4 MBE filler rectangle
# around the stadium by hugging the outer edge of the preserved MBE collar.

from src.rteg_mbe_body_center_pad import (
    MbeBodyCenterPadConfig,
    build_mbe_body_center_pads,
    mbe_body_center_pad_overview_rows,
)

center_pad_cfg = MbeBodyCenterPadConfig()

center_pad_body = build_mbe_body_center_pads(
    all_roles,
    all_classify,
    all_mte,
    layermap,
    center_pad_cfg,
)

from src.rteg_mbe_body import merge_mbe_bodies

all_mbe_body = merge_mbe_bodies(all_mbe_body, center_pad_body)

center_pad_df = pd.DataFrame(
    mbe_body_center_pad_overview_rows(center_pad_body, inst_names=inst_names)
).sort_values("index")
display(center_pad_df)
body_drawn = int(center_pad_df["n_pieces"].sum())
print(f"Drew MBE ground filler for {body_drawn} polygon piece(s) (center_pad)")


## Final export ΓÇö complete routed RTEG (steps 4ΓÇô6.3)

**~30 s read**

**Call:** `export_full_rteg_gds(rteg_assemblies, all_mte, ROUTE_OUTPUTS, layermap=..., parent=..., mbe_extensions=all_mbe, mbe_bodies=all_mbe_body)`

Run this cell **after** steps 6.2 and 6.3 so `all_mbe_body` includes both routing styles (collar_extend + center_pad).

**Output:** `draft_output/route_outputs/{parent}_RTEG1_{index:02d}_{inst_name}.gds` ΓÇö one file per resonator with die frame, PPD, resonator, MTE routes, and MBE signal/ground from every pipeline step. Open in KLayout with the sidecar `.lyp`.


In [ ]:
# Final export — all pipeline geometry in one GDS per resonator.

from src.export_gds import export_summary_df
from src.rteg_mte_extensions import export_full_rteg_gds

ROUTE_OUTPUTS.mkdir(parents=True, exist_ok=True)
final_export_df = export_summary_df(
    export_full_rteg_gds(
        rteg_assemblies,
        all_mte,
        ROUTE_OUTPUTS,
        layermap=layermap,
        parent=parent,
        mbe_extensions=all_mbe,
        mbe_bodies=all_mbe_body,
    )
)

print(f"Exported {len(final_export_df)} complete RTEG GDS files to {ROUTE_OUTPUTS}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(final_export_df)
